## 1) Initialization and Config

Imports, project path setup, global configuration, and runtime constants.

In [ ]:
from __future__ import annotations

import gc
import logging
import os
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import torch
import tqdm
import wandb
from dotenv import load_dotenv
from torch.utils.data import DataLoader
from torchinfo import summary

# Resolve project root robustly for notebook execution from either repo root or this folder.
if (Path.cwd() / "vae_features").exists():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "train" and (Path.cwd().parent / "utils").exists():
    PROJECT_ROOT = Path.cwd().parents[1]
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from vae_features.loss.hyperSphericalLoss import HypersphericalVAELoss
from vae_features.model.feedForwardVae import FeedForwardVAE
from vae_features.model.graphVae import GraphVAE
from vae_features.train.latent_visualization import save_fixed_latent_projection_with_images
from vae_features.train.mhr_pose_dataset import MHRPoseDataset, MHRBatch
from vae_features.train.reconstruction_visualization import save_reconstruction_visualizations
from vae_features.utils.angleFormat import rotation_6d_to_euler
from vae_features.utils.feedForward import Norm
from vae_features.utils.skeletonFormat import SkeletonFormat

CONFIG = {
    "NUM_EPOCHS": 20,
    "BATCH_SIZE": 128,
    "LEARNING_RATE": 1e-3,
    "WEIGHT_DECAY": 1e-8,
    "USE_6D_ROTATIONS": False,
    "USE_GRAPH_VAE": True,
    "USE_VERTEX_SUPERVISION": False,

    # Scheduler parameters
    "WARMUP_EPOCHS": 2,
    "DECAY_EPOCHS": 30,

    # To avoid gradient explosion. Set to 1 to disable
    "GRAD_ACCUMULATION_STEPS": 1,
    # Clip gradients to avoid explosion
    "CLIP_GRADIENTS": False,
    "EMPTY_CACHE_AFTER_BATCH": False,

    "KL_WEIGHT": 25.0,
    "NUM_EPOCHS_TO_FULL_KL": 5,
    "VERTEX_LOSS_WEIGHT": 1.0,

    "DATA_DIR": str(PROJECT_ROOT / "data"),
    "TRAIN_RATIO": 0.9,
    "TOTAL_DATAPOINTS": 50_000,

    "USE_WANDB": True,
    "WANDB_RUN_NAME": "VAE_MHR_train",
    "WANDB_PREVIOUS_RUN_ID": None,
    "WANDB_PROJECT_NAME": "multimodal_2025/Art_ML",

    # Validation artifacts
    "NUM_PCA_SAMPLES": 100,
    "NUM_RECONSTRUCTION_SAMPLES": 6,

    # Model hyperparameters
    "GRAPH_NUM_LAYERS": 4,
    "GRAPH_JOINT_EMBED_DIM": 128,
    "GRAPH_BONE_EMBED_DIM": 64,
    "GRAPH_DECODER_JOINT_EMBED_DIM": 64,
    "GRAPH_NUM_HEADS": 8,
    "GRAPH_BOTTLENECK_DIM": 256,
    "GRAPH_DROPOUT": 0.1,

    "FF_ENCODER_SIZES": [381, 512, 256, 128],
    "FF_DROPOUT": 0.1,
    "FF_USE_RESIDUALS": True,
    "FF_NORMALIZATION": "layer",

    "CHECKPOINT_BASENAME": "mhr_vae",
    "SEED": 42,
}

TRAIN_DIR = PROJECT_ROOT / "vae_features" / "train"
SKELETON_JSON_PATH = TRAIN_DIR / "mhr_skeleton_format.json"
JOINT_NAMES_JSON_PATH = TRAIN_DIR / "joint_names.json"
PARQUET_PATH = Path(CONFIG["DATA_DIR"]) / "processed_poses.parquet"
MHR_MODEL_PT_PATH = PROJECT_ROOT / "checkpoints" / "sam3d" / "dinov3" / "assets" / "mhr_model.pt"
VALIDATION_DIR = TRAIN_DIR / "validation"
PCA_DIR = VALIDATION_DIR / "PCA"
RECONSTRUCTION_DIR = VALIDATION_DIR / "reconstruction"
CHECKPOINT_DIR = TRAIN_DIR / "checkpoints"

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("train_vae")
logger.info(f"Using device: {DEVICE}")

## 2) Reproducibility and Output Paths

Set random seeds, create output directories, and validate data availability.

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_output_dirs():
    for path in [PCA_DIR, RECONSTRUCTION_DIR, CHECKPOINT_DIR]:
        path.mkdir(parents=True, exist_ok=True)


set_seed(CONFIG["SEED"])
make_output_dirs()

if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Missing parquet file: {PARQUET_PATH}")

logger.info(f"Parquet file: {PARQUET_PATH}")

## 3) Dataset and DataLoaders

Load skeleton format + parquet dataset and create train/validation data loaders.

In [ ]:
skeleton_format = SkeletonFormat.from_json_file(SKELETON_JSON_PATH)

dataset = MHRPoseDataset(
    parquet_path=PARQUET_PATH,
    skeleton_format=skeleton_format,
    joint_names_path=JOINT_NAMES_JSON_PATH,
    data_root=CONFIG["DATA_DIR"],
    max_samples=CONFIG["TOTAL_DATAPOINTS"],
    device=torch.device(DEVICE),
)

total_samples = len(dataset)
num_train = int(total_samples * CONFIG["TRAIN_RATIO"])
num_val = total_samples - num_train

if num_train <= 0 or num_val <= 0:
    raise ValueError(
        f"Invalid split from {total_samples} samples with TRAIN_RATIO={CONFIG['TRAIN_RATIO']}"
    )

generator = torch.Generator().manual_seed(CONFIG["SEED"])
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [num_train, num_val],
    generator=generator,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    num_workers=0,
    pin_memory=DEVICE == "cuda",
    collate_fn=dataset.collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,
    num_workers=0,
    pin_memory=DEVICE == "cuda",
    collate_fn=dataset.collate_fn,
)

logger.info(f"Train samples: {len(train_dataset)}")
logger.info(f"Val samples: {len(val_dataset)}")

## 4) Model, Loss, Optimizer, and Scheduler

Initialize GraphVAE/FeedForwardVAE, hyperspherical loss, optimizer, and warmup+cosine schedulers.

In [ ]:
if not CONFIG["USE_GRAPH_VAE"] and CONFIG["USE_6D_ROTATIONS"]:
    raise ValueError("USE_6D_ROTATIONS=True is currently supported only with GraphVAE.")

if CONFIG["USE_GRAPH_VAE"]:
    model = GraphVAE(
        skeleton_format=skeleton_format,
        num_layers=CONFIG["GRAPH_NUM_LAYERS"],
        joint_embedding_dimension=CONFIG["GRAPH_JOINT_EMBED_DIM"],
        bone_embedding_dimension=CONFIG["GRAPH_BONE_EMBED_DIM"],
        decoder_joint_embedding_dimension=CONFIG["GRAPH_DECODER_JOINT_EMBED_DIM"],
        num_attention_heads=CONFIG["GRAPH_NUM_HEADS"],
        bottleneck_dimensions=CONFIG["GRAPH_BOTTLENECK_DIM"],
        bottleneck_activation=torch.nn.GELU(),
        dropout=CONFIG["GRAPH_DROPOUT"],
        use_6d_rotation_format=CONFIG["USE_6D_ROTATIONS"],
    )
else:
    norm = Norm.LAYER if CONFIG["FF_NORMALIZATION"].lower() == "layer" else Norm.BATCH
    model = FeedForwardVAE(
        skeletonFormat=skeleton_format,
        encoderSizes=CONFIG["FF_ENCODER_SIZES"],
        dropout=CONFIG["FF_DROPOUT"],
        use_residuals=CONFIG["FF_USE_RESIDUALS"],
        activation=torch.nn.GELU(),
        normalization=norm,
    )

model = model.to(DEVICE).float()
logger.info(f"Model initialized: {type(model).__name__}")

In [ ]:
MODEL_PARAM_COUNT = sum(p.numel() for p in model.parameters())
logger.info(f"Model parameter count: {MODEL_PARAM_COUNT:,}")

class _ModelSummaryWrapper(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        if CONFIG["USE_GRAPH_VAE"]:
            _, latent, _ = self.base_model.encode(x)
            return self.base_model.decode(latent)
        flat = x.reshape(x.shape[0], -1)
        _, latent, _ = self.base_model.encode(flat)
        return self.base_model.decode(latent)


summary_wrapper = _ModelSummaryWrapper(model)
summary_input_shape = (1, skeleton_format.get_joint_count(), 3)
summary(summary_wrapper, input_size=summary_input_shape, device=DEVICE)

In [ ]:
criterion = HypersphericalVAELoss(
    use_6d_rotation_format=CONFIG["USE_6D_ROTATIONS"],
    use_vertex_supervision=CONFIG["USE_VERTEX_SUPERVISION"],
    mhr_model_path=str(MHR_MODEL_PT_PATH) if CONFIG["USE_VERTEX_SUPERVISION"] else None,
    vertex_loss_weight=float(CONFIG["VERTEX_LOSS_WEIGHT"]),
).to(DEVICE)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["LEARNING_RATE"],
    weight_decay=CONFIG["WEIGHT_DECAY"],
)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=CONFIG["WARMUP_EPOCHS"],
)
decay_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["DECAY_EPOCHS"],
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, decay_scheduler],
    milestones=[CONFIG["WARMUP_EPOCHS"]],
)


def kl_schedule(epoch: int) -> float:
    epoch = epoch + 1
    if epoch < CONFIG["NUM_EPOCHS_TO_FULL_KL"]:
        return 0.0
    if epoch >= 2 * CONFIG["NUM_EPOCHS_TO_FULL_KL"]:
        return float(CONFIG["KL_WEIGHT"])
    return float(CONFIG["KL_WEIGHT"]) * (
        epoch / (2 * CONFIG["NUM_EPOCHS_TO_FULL_KL"])
    )

## 5) Training Loop

Define forward/reconstruction behavior and train one epoch with KL scheduling and gradient controls.

In [ ]:
def vae_output_to_mhr_params(
    reconstructed_joint_output: torch.Tensor,
    original_raw: torch.Tensor,
    inverse_reorder_indices: torch.Tensor,
    post_processor,
    use_6d_rotations: bool,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Differentiable helper that maps VAE outputs back to raw and MHR parameters.
    """
    if reconstructed_joint_output.ndim == 2:
        reconstructed_joint_output = reconstructed_joint_output.view(
            reconstructed_joint_output.shape[0],
            inverse_reorder_indices.shape[0],
            3,
        )

    if reconstructed_joint_output.ndim != 3:
        raise ValueError(
            f"Expected reconstructed output with shape (B, J, D), got {reconstructed_joint_output.shape}"
        )

    if use_6d_rotations:
        if reconstructed_joint_output.shape[-1] != 6:
            raise ValueError(
                f"Expected 6D reconstructed rotations when use_6d_rotations=True, got {reconstructed_joint_output.shape}"
            )
        euler_joint_angles = rotation_6d_to_euler(reconstructed_joint_output)
    else:
        if reconstructed_joint_output.shape[-1] != 3:
            raise ValueError(
                f"Expected Euler reconstructed rotations with last dim 3, got {reconstructed_joint_output.shape}"
            )
        euler_joint_angles = reconstructed_joint_output

    inverse_idx = inverse_reorder_indices.to(euler_joint_angles.device)
    euler_joint_angles_mhr_order = euler_joint_angles[:, inverse_idx, :]

    patched_raw, reconstructed_mhr_params = post_processor.joint_angles_to_mhr_parameters(
        joint_angles_mhr_order=euler_joint_angles_mhr_order,
        base_raw=original_raw,
    )
    return patched_raw, reconstructed_mhr_params

In [ ]:
def forward_and_reconstruct(batch_angles: torch.Tensor):
    if CONFIG["USE_GRAPH_VAE"]:
        latent_distribution, latent, _ = model.encode(batch_angles)
        reconstruction = model.decode(latent)
        target = batch_angles
        return latent_distribution, latent, reconstruction, target

    flat_input = batch_angles.reshape(batch_angles.shape[0], -1)
    latent_distribution, latent, _ = model.encode(flat_input)
    reconstruction = model.decode(latent)
    target = flat_input
    return latent_distribution, latent, reconstruction, target


def train_epoch(epoch: int) -> dict[str, float]:
    model.train()
    progress = tqdm.tqdm(train_loader, desc=f"Train Epoch {epoch + 1}")

    total_loss = 0.0
    batches_seen = 0

    optimizer.zero_grad()
    for batch_idx, batch in enumerate(progress):
        batch = batch
        joint_angles = batch.joint_angles.to(DEVICE)

        if torch.isnan(joint_angles).any():
            logger.warning(f"NaN in train batch {batch_idx}, skipping")
            continue

        latent_distribution, _, reconstruction, target = forward_and_reconstruct(joint_angles)

        reconstructed_mhr_params = None
        target_mhr_params = None
        if CONFIG["USE_VERTEX_SUPERVISION"]:
            _, reconstructed_mhr_params = vae_output_to_mhr_params(
                reconstructed_joint_output=reconstruction,
                original_raw=batch.joint_data.raw,
                inverse_reorder_indices=dataset.inverse_reorder_indices,
                post_processor=dataset.post_processor,
                use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
            )
            target_mhr_params = torch.stack(
                [meta.mhr_parameters for meta in batch.metadata], dim=0
            ).to(DEVICE)

        loss = criterion(
            predicted_joint_angles=reconstruction,
            label_joint_angles=target,
            latent_distributions=latent_distribution,
            kl_weight=kl_schedule(epoch),
            reconstructed_mhr_params=reconstructed_mhr_params,
            target_mhr_params=target_mhr_params,
        )

        if torch.isnan(loss):
            logger.warning(f"NaN loss in train batch {batch_idx}, skipping")
            optimizer.zero_grad()
            continue

        scaled_loss = loss / CONFIG["GRAD_ACCUMULATION_STEPS"]
        scaled_loss.backward()

        has_nan_grad = False
        for _, param in model.named_parameters():
            if param.grad is not None and torch.isnan(param.grad).any():
                has_nan_grad = True
                break

        if has_nan_grad:
            logger.warning(f"NaN gradients in train batch {batch_idx}, skipping update")
            optimizer.zero_grad()
            continue

        if (batch_idx + 1) % CONFIG["GRAD_ACCUMULATION_STEPS"] == 0:
            if CONFIG["CLIP_GRADIENTS"]:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        if CONFIG["EMPTY_CACHE_AFTER_BATCH"] and DEVICE == "cuda":
            torch.cuda.empty_cache()
            gc.collect()

        total_loss += float(loss.item())
        batches_seen += 1
        progress.set_postfix({"loss": total_loss / max(batches_seen, 1)})

    if len(train_loader) % CONFIG["GRAD_ACCUMULATION_STEPS"] != 0:
        if CONFIG["CLIP_GRADIENTS"]:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = total_loss / max(batches_seen, 1)
    return {
        "train_loss": avg_loss,
        "kl_weight": kl_schedule(epoch),
        "learning_rate": scheduler.get_last_lr()[0],
    }

## 6) Validation: Latent Projection + Reconstruction Visualization

Build validation embeddings, write latent-space image projections on a fixed subset, and render side-by-side original vs original-MHR vs reconstructed-MHR outputs.

In [ ]:
def _save_latent_projection(embeddings: np.ndarray, metadata: list[dict[str, Any]], epoch: int) -> str | None:
    out_path = PCA_DIR / f"epoch_{epoch + 1}_latent_projection.png"
    saved_path = save_fixed_latent_projection_with_images(
        embeddings=embeddings,
        metadata=metadata,
        output_path=out_path,
        title=f"Latent projection (epoch {epoch + 1})",
        num_samples=int(CONFIG["NUM_PCA_SAMPLES"]),
        seed=int(CONFIG["SEED"]),
        data_dir=CONFIG["DATA_DIR"],
        method="pca",
    )
    if saved_path is None:
        logger.warning("Not enough valid images for latent projection")
    return saved_path


def _save_reconstruction_visualizations(samples: list[dict[str, Any]], epoch: int) -> list[str]:
    saved_paths = save_reconstruction_visualizations(
        samples=samples,
        epoch=epoch,
        output_dir=RECONSTRUCTION_DIR,
        project_root=PROJECT_ROOT,
        data_dir=CONFIG["DATA_DIR"],
        num_samples=int(CONFIG["NUM_RECONSTRUCTION_SAMPLES"]),
        seed=int(CONFIG["SEED"] + 17),
    )
    return saved_paths


def validate_epoch(epoch: int) -> dict[str, Any]:
    model.eval()

    total_loss = 0.0
    batches_seen = 0
    all_embeddings: list[np.ndarray] = []
    all_metadata: list[dict[str, Any]] = []
    reconstruction_samples: list[dict[str, Any]] = []

    with torch.no_grad():
        progress = tqdm.tqdm(val_loader, desc=f"Val Epoch {epoch + 1}")
        for batch_idx, batch in enumerate(progress):
            batch = batch  # type: MHRBatch
            joint_angles = batch.joint_angles.to(DEVICE)

            if torch.isnan(joint_angles).any():
                logger.warning(f"NaN in val batch {batch_idx}, skipping")
                continue

            latent_distribution, latent, reconstruction, target = forward_and_reconstruct(joint_angles)

            reconstructed_mhr_params = None
            target_mhr_params = None
            if CONFIG["USE_VERTEX_SUPERVISION"]:
                _, reconstructed_mhr_params = vae_output_to_mhr_params(
                    reconstructed_joint_output=reconstruction,
                    original_raw=batch.joint_data.raw,
                    inverse_reorder_indices=dataset.inverse_reorder_indices,
                    post_processor=dataset.post_processor,
                    use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
                )
                target_mhr_params = torch.stack(
                    [meta.mhr_parameters for meta in batch.metadata], dim=0
                ).to(DEVICE)

            loss = criterion(
                predicted_joint_angles=reconstruction,
                label_joint_angles=target,
                latent_distributions=latent_distribution,
                kl_weight=kl_schedule(epoch),
                reconstructed_mhr_params=reconstructed_mhr_params,
                target_mhr_params=target_mhr_params,
            )
            if torch.isnan(loss):
                continue

            total_loss += float(loss.item())
            batches_seen += 1
            progress.set_postfix({"loss": total_loss / max(batches_seen, 1)})

            all_embeddings.extend(latent.detach().cpu().numpy())
            for meta in batch.metadata:
                all_metadata.append(
                    {
                        "image_path": meta.image_path,
                        "image_path_abs": meta.image_path_abs,
                    }
                )

            _, reconstructed_mhr_params = vae_output_to_mhr_params(
                reconstructed_joint_output=reconstruction,
                original_raw=batch.joint_data.raw,
                inverse_reorder_indices=dataset.inverse_reorder_indices,
                post_processor=dataset.post_processor,
                use_6d_rotations=CONFIG["USE_6D_ROTATIONS"],
            )

            for meta, reconstructed_mhr in zip(batch.metadata, reconstructed_mhr_params, strict=False):
                reconstruction_samples.append(
                    {
                        "image_path": meta.image_path,
                        "image_path_abs": meta.image_path_abs,
                        "pred_cam": meta.pred_cam,
                        "pred_cam_t": meta.pred_cam_t,
                        "focal_length": meta.focal_length,
                        "pred_keypoints_2d": meta.pred_keypoints_2d,
                        "original_mhr_parameters": meta.mhr_parameters,
                        "reconstructed_mhr_parameters": reconstructed_mhr,
                    }
                )

            if CONFIG["EMPTY_CACHE_AFTER_BATCH"] and DEVICE == "cuda":
                torch.cuda.empty_cache()
                gc.collect()

    embeddings = np.array(all_embeddings) if all_embeddings else np.zeros((0, 0), dtype=np.float32)
    pca_path = _save_latent_projection(embeddings, all_metadata, epoch)
    reconstruction_paths = _save_reconstruction_visualizations(reconstruction_samples, epoch)

    return {
        "val_loss": total_loss / max(batches_seen, 1),
        "pca_path": pca_path,
        "reconstruction_paths": reconstruction_paths,
    }

## 7) WandB and Full Training Run

Initialize WandB, run epoch training/validation, log artifacts, and save checkpoints.

In [ ]:
load_dotenv()
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

run = None
if CONFIG["USE_WANDB"]:
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY)
    else:
        logger.warning("WANDB_API_KEY missing; wandb may prompt for login")

    resume_logging = CONFIG["WANDB_PREVIOUS_RUN_ID"] is not None
    if resume_logging:
        run = wandb.init(
            settings=wandb.Settings(symlink=False),
            id=CONFIG["WANDB_PREVIOUS_RUN_ID"],
            resume="must",
            project=CONFIG["WANDB_PROJECT_NAME"],
            config=CONFIG,
        )
    else:
        run = wandb.init(
            name=CONFIG["WANDB_RUN_NAME"],
            reinit=True,
            project=CONFIG["WANDB_PROJECT_NAME"],
            config=CONFIG,
        )

    wandb.log({"model_param_count": MODEL_PARAM_COUNT})

best_val_loss = float("inf")
history: list[dict[str, float]] = []

for epoch in range(CONFIG["NUM_EPOCHS"]):
    logger.info(f"Epoch {epoch + 1}/{CONFIG['NUM_EPOCHS']}")
    train_metrics = train_epoch(epoch)
    val_report = validate_epoch(epoch)

    scheduler.step()

    epoch_metrics = {
        **train_metrics,
        "val_loss": val_report["val_loss"],
        "epoch": epoch + 1,
    }
    history.append(epoch_metrics)

    logger.info(
        f"train_loss={epoch_metrics['train_loss']:.6f} val_loss={epoch_metrics['val_loss']:.6f} "
        f"lr={scheduler.get_last_lr()[0]:.6f}"
    )

    latest_ckpt = CHECKPOINT_DIR / f"{CONFIG['CHECKPOINT_BASENAME']}_latest.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "history": history,
            "epoch": epoch + 1,
        },
        latest_ckpt,
    )

    if epoch_metrics["val_loss"] < best_val_loss:
        best_val_loss = epoch_metrics["val_loss"]
        best_ckpt = CHECKPOINT_DIR / f"{CONFIG['CHECKPOINT_BASENAME']}_best.pt"
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": CONFIG,
                "history": history,
                "epoch": epoch + 1,
            },
            best_ckpt,
        )
        logger.info(f"Saved best checkpoint: {best_ckpt}")

    if run is not None:
        log_payload: dict[str, Any] = {
            "train_loss": epoch_metrics["train_loss"],
            "val_loss": epoch_metrics["val_loss"],
            "kl_weight": epoch_metrics["kl_weight"],
            "learning_rate": scheduler.get_last_lr()[0],
            "epoch": epoch + 1,
        }

        if val_report["pca_path"] is not None and Path(val_report["pca_path"]).exists():
            log_payload["latent_projection"] = wandb.Image(val_report["pca_path"])

        for i, recon_path in enumerate(val_report.get("reconstruction_paths", [])[:3]):
            if Path(recon_path).exists():
                log_payload[f"reconstruction_{i}"] = wandb.Image(recon_path)

        wandb.log(log_payload)

if run is not None:
    run.finish()

logger.info("Training complete.")
logger.info(f"Checkpoints saved under: {CHECKPOINT_DIR}")